# N05 · Pipeline Parallel Bubble：为什么流水线会空转？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

Pipeline Parallel（PP）把模型按层切到不同 GPU/stage 上。它解决的是“模型太深/太大，单卡或单组 TP 放不下”的问题。但 PP 会出现 bubble：有些 stage 在等待前一个 stage 产出，或者等待后一个 stage 反传。

本节目标：理解 microbatch 如何降低 bubble，以及为什么 microbatch 不是越多越好。


## 学习地图与版本说明（截至 2026-04-30）

本节从“切层”理解 PP：TP 是把层内矩阵切开，PP 是把模型深度方向切成 stage。PP 的核心难点不是能不能切，而是如何让 stage 尽量少等待。microbatch 数量越少，流水线 warmup/cooldown 的空洞越明显；microbatch 数量增加可以降低 bubble，但也会改变每个 microbatch 的显存、kernel 粒度和调度开销。

版本上，本教程参考 GPipe、Megatron-LM pipeline parallel 论文，以及 NVIDIA Megatron Core 最新并行策略文档。现代 Megatron 系列实现还包含 1F1B、virtual/interleaved pipeline、custom pipeline layout 等机制；这些机制名字会随框架演进，但目标始终是减少 bubble、平衡 stage 负载、控制 activation 与通信开销。

学完本节，你应该能把 global batch、micro batch、gradient accumulation steps 区分开；能解释为什么 stage 数增加不一定更快；能用课程 L05 的指标判断是 bubble 大、stage 不均衡、通信慢还是 microbatch 过小导致 kernel 效率下降。


## 1. PP 的第一性原理：切层，不切矩阵

Tensor parallel 切的是单层内部的矩阵；pipeline parallel 切的是模型层序列：

```text
Stage 0: layers 0..7
Stage 1: layers 8..15
Stage 2: layers 16..23
Stage 3: layers 24..31
```

一个 batch 会被拆成多个 microbatch，像流水线上的零件一样依次经过各 stage。没有 microbatch 时，后面的 stage 一开始没活干，前面的 stage 结束后也会等反传，这些空闲时间就是 bubble。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def bubble_ratio(stages, microbatches):
    # GPipe 风格简化估算：总时隙约 m + s - 1，有效工作 m；按每个 stage 归一化
    return (stages - 1) / (microbatches + stages - 1)

rows = []
for stages in [2, 4, 8]:
    for microbatches in [1, 2, 4, 8, 16, 32]:
        rows.append({"stages": stages, "microbatches": microbatches, "bubble_ratio": bubble_ratio(stages, microbatches)})

df = pd.DataFrame(rows)
display(df.round(3))
for s, part in df.groupby("stages"):
    plt.plot(part["microbatches"], part["bubble_ratio"], marker="o", label=f"stages={s}")
plt.xscale("log", base=2)
plt.xlabel("microbatches")
plt.ylabel("bubble ratio 粗略估算")
plt.title("microbatch 增加时 bubble 下降")
plt.legend()
plt.show()


## 2. microbatch、global batch、gradient accumulation 的关系

Pipeline 中常见三个 batch 概念：

- **micro batch size**：单个 microbatch 每张卡处理多少样本。
- **num microbatches / gradient accumulation steps**：一次 optimizer step 前累计多少个 microbatch。
- **global batch size**：一次 optimizer step 覆盖的全局样本。

简化公式：

```text
global_batch = micro_batch_size × data_parallel_size × num_microbatches
```

PP 中增加 microbatches 可以降低 bubble，但也会改变 optimizer step 的统计语义，除非你同步调整 micro batch size 或 global batch。


In [ ]:
def batch_formula(micro_batch_size, data_parallel_size, num_microbatches):
    return micro_batch_size * data_parallel_size * num_microbatches

pd.DataFrame([
    {"micro_batch_size": 1, "DP": 8, "num_microbatches": 8, "global_batch": batch_formula(1, 8, 8)},
    {"micro_batch_size": 2, "DP": 8, "num_microbatches": 8, "global_batch": batch_formula(2, 8, 8)},
    {"micro_batch_size": 1, "DP": 8, "num_microbatches": 16, "global_batch": batch_formula(1, 8, 16)},
])


## 3. 为什么 microbatch 不是越多越好？

增加 microbatch 会降低 bubble，但也可能带来：

- 每个 microbatch 太小，matmul 形状变差，GPU 利用率下降。
- 调度开销增加。
- activation 保存/重算策略变化。
- global batch 改变，优化器语义变化。
- pipeline stage 间通信次数增加。

因此 PP 调参不是单变量最大化，而是平衡 bubble、kernel 效率、显存、通信、收敛语义。


## 4. 1F1B 和 interleaved pipeline 的直觉

GPipe 风格通常先跑完 forward 再 backward，激活保存压力较高。1F1B（one-forward-one-backward）在填满流水线后交替 forward/backward，可以降低激活驻留时间。

Megatron 进一步有 interleaved pipeline schedule，把每个物理 stage 切成多个 virtual stage，以降低 bubble 并改善负载平衡。但这也增加调度和配置复杂度。


In [ ]:
def pp_recommendation(stages, microbatches, per_microbatch_tokens):
    br = bubble_ratio(stages, microbatches)
    warnings = []
    if microbatches < stages:
        warnings.append("microbatches 少于 stages，bubble 可能很高")
    if per_microbatch_tokens < 1024:
        warnings.append("单 microbatch tokens 太少，kernel 可能不饱和")
    if br > 0.25:
        warnings.append("bubble_ratio 偏高，考虑增加 microbatches 或 interleaving")
    return {"stages": stages, "microbatches": microbatches, "bubble_ratio": round(br, 3), "warnings": warnings or ["配置看起来合理"]}

for cfg in [(4, 2, 2048), (4, 16, 512), (8, 32, 4096)]:
    print(pp_recommendation(*cfg))


## 5. 与本课程的连接

- L02 的 `toy_pipeline_two_stage.py` 做两阶段 pipeline 最小实验。
- L05 会把 PP 与 TP/recompute/MFU 一起讨论。
- L12 Capstone 中，迁移到 8×H200 时需要说明 PP 是否进入配置，以及 global batch 是否保持。


## 6. 企业面试/工程判断痛点题（带答案）

### 题 1：Pipeline parallel 解决的是参数显存还是激活显存？

**答案解析：** 主要通过切层降低每个 stage 持有的参数和部分激活压力，但激活仍与 microbatch、schedule、checkpointing 相关。它不是单独的“激活显存优化”。

### 题 2：为什么 PP stage 数增加后吞吐可能下降？

**答案解析：** stage 增加会提高 bubble，增加 stage 间通信，并可能导致负载不均。microbatch 不足时，很多 GPU 会等待。

### 题 3：增加 microbatches 降低 bubble，会不会改变训练语义？

**答案解析：** 会。若 micro batch size 和 DP 不变，num_microbatches 增加会增大 global batch。需要同步调整或明确报告训练语义变化。

### 题 4：TP 和 PP 应该如何放在硬件拓扑上？

**答案解析：** TP 通信更频繁，通常尽量放在高速互联域内（如 NVLink/NVSwitch）。PP stage 间通信较粗粒度，但也要考虑跨节点延迟和带宽。

### 题 5：看到 GPU 利用率低，如何判断是不是 pipeline bubble？

**答案解析：** 看 timeline：不同 stage 是否周期性等待、空闲是否与 microbatch schedule 对齐；同时计算 bubble ratio，并尝试增加 microbatches 做单变量验证。


## 参考资料

- Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM: https://huggingface.co/papers/2104.04473
- GPipe paper: https://arxiv.org/abs/1811.06965
- FairScale Pipeline Parallelism deep dive: https://fairscale.readthedocs.io/en/latest/deep_dive/pipeline_parallelism.html
